# model

> Compile the PP-OCRv6 medium modules and bind their checkpoints.


In [ ]:
#| default_exp model


#| export
The detector is one graph per input shape, because both sides are fixed and must be multiples of 32. The recognizer is one graph, 48×320. A cache directory of compiled MEFs is reused; an empty directory is where a new compilation is written.


In [ ]:
#| hide
from nbdev import show_doc


In [ ]:
#| export
#| notest
from __future__ import annotations

from pathlib import Path

from max.driver import CPU
from max.engine import InferenceSession
from max.graph import Graph, TensorType

from solid_lens.config import DEVICE, DTYPE, RecognizerConfig
from solid_lens.core import repo_root
from solid_lens.detector import PPOCRV6Detector
from solid_lens.recognizer import PPOCRV6Recognizer
from solid_lens.weights import load_checkpoint


In [ ]:
#| export
#| notest
ROOT = repo_root()


In [ ]:
#| export
#| notest
def _session(cache_dir: Path | None) -> InferenceSession:
    kwargs = {}
    if cache_dir is not None:
        cache_dir.mkdir(parents=True, exist_ok=True)
        if any(cache_dir.iterdir()):
            kwargs["precompiled_mefs"] = cache_dir
        else:
            kwargs["export_mefs"] = cache_dir
    return InferenceSession(devices=[CPU()], **kwargs)


A non-empty cache is loaded. An empty one receives the compiled model. The directory is the cache key the callers choose, including the detector's height and width in the name.


In [ ]:
#| export
#| notest
def _compile(module, shape: list[int], name: str, cache_dir: Path | None):
    graph = Graph(name, input_types=[TensorType(DTYPE, shape, device=DEVICE)])
    with graph:
        graph.output(module(graph.inputs[0].tensor))
    weights = module.state_dict()
    return _session(cache_dir).load(graph, weights_registry=weights)


In [ ]:
#| export
#| notest
def compile_detector(
    size: int | None = None,
    cache_dir: Path | None = None,
    *,
    height: int | None = None,
    width: int | None = None,
):
    "Compile the detector for one input shape."
    if height is None and width is None:
        if size is None:
            raise ValueError("pass size, or height and width")
        height = width = size
    elif height is None or width is None:
        raise ValueError("height and width are set together")
    if height % 32 or width % 32:
        raise ValueError("detector sides must be multiples of 32")
    module = PPOCRV6Detector()
    load_checkpoint(module, ROOT / "models" / "medium_det.onnx")
    return _compile(
        module,
        [1, 3, height, width],
        f"ppocrv6_medium_det_{height}x{width}",
        cache_dir,
    )


Pass `size` for a square, or `height` and `width` for the rectangle `detection_hw` produces. Both sides must be multiples of 32, which `detection_hw` already guarantees.


In [ ]:
#| export
#| notest
def compile_recognizer(cache_dir: Path | None = None):
    config = RecognizerConfig()
    module = PPOCRV6Recognizer(config)
    load_checkpoint(module, ROOT / "models" / "medium_rec.onnx")
    return _compile(
        module,
        [1, 3, config.image_height, config.image_width],
        "ppocrv6_medium_rec",
        cache_dir,
    )
